<a href="https://colab.research.google.com/github/warrensuca/chinese-recipe-api/blob/main/lkk_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lee Kum Kee Recipe Scraper

This notebook scrapes recipe data from the Lee Kum Kee recipe website.

For each recipe, we collect:

- Name
- Ingredients
- Seasoning
- Marinade
- Sauce Mix
- Procedure

The final dataset is stored as a Pandas DataFrame and exported to CSV.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display

## Create Session

The goal is to reuse connections

In [ ]:
session = requests.Session()

session.headers.update({
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
})

print("Session initialized")

In [ ]:
def scrape_recipe(url):

    headings = [
        "Name",
        "Ingredients",
        "Seasoning",
        "Marinade",
        "Sauce Mix",
        "Procedure"
    ]

    response = session.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    recipe_data = {
        "Name": url.split("/")[-1]
    }

    boxes = soup.find_all("div", class_="side-box")

    for box in boxes:

        heading = box.find("h3")

        if heading:

            items = [
                item.text.strip()
                for item in box.find_all("li")
            ]

            recipe_data[heading.text.strip()] = items

    steps = soup.find_all("li", class_="step-box")

    recipe_data["Procedure"] = [
        step.text.strip()
        for step in steps
    ]

    row = []

    for heading in headings:
        row.append(recipe_data.get(heading, []))

    return row

In [ ]:
page_num = 1

url = f"https://usa.lkk.com/en/recipes?page={page_num}"

print(f"Fetching page {page_num}")

response = session.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

recipes = soup.find_all("div", class_="recipe-item")

print(f"Found {len(recipes)} recipes")

In [ ]:
recipe_urls = []

for recipe in recipes:

    link = recipe.find("a")

    if link and "href" in link.attrs:

        recipe_urls.append(
            f"https://usa.lkk.com{link['href']}"
        )

print(f"Collected {len(recipe_urls)} URLs")
print(recipe_urls[:5])

#Test scraping a single recipe

In [ ]:
sample_recipe = scrape_recipe(recipe_urls[0])

sample_recipe

#Scrape entire site

In [ ]:
pages = 151

data = []

for page in range(1, pages + 1):

    print(f"\nPage {page}/{pages}")

    try:

        url = f"https://usa.lkk.com/en/recipes?page={page}"

        response = session.get(url, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        recipes = soup.find_all(
            "div",
            class_="recipe-item"
        )

        print(f"Found {len(recipes)} recipes")

        for recipe in recipes:

            link = recipe.find("a")

            if link and "href" in link.attrs:

                recipe_url = (
                    f"https://usa.lkk.com{link['href']}"
                )

                try:

                    recipe_data = scrape_recipe(recipe_url)

                    data.append(recipe_data)

                    print(
                        f"  ✓ {recipe_url.split('/')[-1]}"
                    )

                except Exception as e:

                    print(
                        f"  ✗ Failed recipe: {e}"
                    )

    except Exception as e:

        print(f"Error on page {page}: {e}")
        break

In [ ]:
columns = [
    "Name",
    "Ingredients",
    "Seasoning",
    "Marinade",
    "Sauce Mix",
    "Procedure"
]

df = pd.DataFrame(data, columns=columns)

print(f"Total recipes scraped: {len(df)}")

display(df.head())

In [ ]:
df.info()

In [ ]:
df.sample(5)

In [ ]:
output_file = "lkk_recipes.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(f"Saved {len(df)} recipes")
print(f"Output file: {output_file}")